# CarryBench Colab Runner

Run the audited JAX/Flax vs PyTorch benchmark and bundle raw results, environment metadata, summaries, and plots.

**Before running:** select a CUDA GPU runtime.

In [ ]:
# CarryBench is public, so no token or GitHub login is required.
%cd /content
!rm -rf /content/CarryBench
!git clone https://github.com/vishalvinjamuri27/CarryBench.git /content/CarryBench
%cd /content/CarryBench
from pathlib import Path
assert Path('pyproject.toml').exists(), 'CarryBench was not cloned correctly.'
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements.txt

## Device Check

In [ ]:
import jax, torch
print('JAX devices:', jax.devices())
print('Torch CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert any(d.platform == 'gpu' for d in jax.devices()), 'Select a GPU runtime and install a CUDA-enabled JAX build'
assert torch.cuda.is_available(), 'PyTorch cannot see the GPU'
!nvidia-smi

## Smoke Test

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], check=True)
subprocess.run([sys.executable, '-m', 'src.train_jax', '--config', 'configs/smoke.yaml'], check=True)
subprocess.run([sys.executable, '-m', 'src.train_torch', '--config', 'configs/smoke.yaml'], check=True)

## Final Experiment Suite

Runs paired 5/6-digit experiments with free-running generation evaluation, answer-only and reversed-answer ablations, eager/SDPA/compiled runtime baselines, extended KV-cache sweeps, and result summaries.

In [ ]:
!./scripts/run_final_experiments.sh

## Inspect And Bundle Results

In [ ]:
!python -m src.summarize_results --results-dir results
!./scripts/export_release_artifacts.sh
!head -40 results/summary_aggregate.md

import glob, os, zipfile

bundle_path = 'results_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(glob.glob('results/**/*', recursive=True) + glob.glob('artifacts/**/*', recursive=True)):
        if os.path.isfile(path):
            zf.write(path, arcname=path)

print('Wrote', bundle_path)